# Walking and Biking Image-Location Points

This notebook displays walking and biking routes and their proposed image-location points from the included saved JSON files. Points were previously generated approximately every **200 feet**, with additional points at turns of **30 degrees or more**.

Running this notebook makes **no Google API calls** and requires **no API key**.


In [ ]:
# Imports and map-building setup. No API calls are made.
import json
from pathlib import Path

import folium

# Match the neighborhood colors used in the original project.
NEIGHBORHOOD_COLORS = {
    "Airport District": "salmon",
    "Downtown SLO": "blue",
    "Foothill": "cadetblue",
    "Laguna": "purple",
    "Laguna Lake": "purple",
    "LoBro": "darkred",
    "LoHI": "green",
    "SLO South": "darkgreen",
    "Mojo": "orange",
    "RR": "brown",
}


def build_point_map(saved_data_path, travel_label):
    """Build a Folium map strictly from saved JSON; no API calls occur here."""
    saved_data = json.loads(Path(saved_data_path).read_text(encoding="utf-8"))
    saved_routes = saved_data["routes"]

    point_map = folium.Map(location=[35.27, -120.66], zoom_start=13)
    route_layer = folium.FeatureGroup(
        name=f"Actual {travel_label.lower()} routes", show=True
    ).add_to(point_map)
    interval_layer = folium.FeatureGroup(
        name="Regular 200-foot points", show=True
    ).add_to(point_map)
    turn_layer = folium.FeatureGroup(
        name="Turn points", show=True
    ).add_to(point_map)
    endpoint_layer = folium.FeatureGroup(
        name="Origins and destinations", show=True
    ).add_to(point_map)

    mapped_points = []
    point_count = 0

    for route in saved_routes:
        route_points = route["route_points"]
        image_locations = route["image_locations"]
        distance_meters = route["distance_meters"]
        duration = route["duration"]
        mapped_points.extend(route_points)
        duration_minutes = round(float(duration.rstrip("s")) / 60)
        route_color = NEIGHBORHOOD_COLORS.get(route["neighborhood"], "gray")
        folium.PolyLine(
            route_points,
            color=route_color,
            weight=6,
            opacity=0.9,
            tooltip=(
                f"{travel_label} to {route['destination_name']} — "
                f"{distance_meters / 1000:.2f} km, about {duration_minutes} min"
            ),
        ).add_to(route_layer)

        for location in image_locations:
            point_count += 1
            point = [location["latitude"], location["longitude"]]
            reason = location["selection_reason"]

            if "turn" in reason:
                color, radius, layer = "orange", 4, turn_layer
            elif "origin" in reason:
                color, radius, layer = "red", 5, endpoint_layer
            elif "destination" in reason:
                color, radius, layer = "black", 5, endpoint_layer
            else:
                color, radius, layer = "cadetblue", 3, interval_layer

            elevation = location.get("elevation_meters")
            resolution = location.get("elevation_resolution_meters")
            if elevation is None:
                elevation_label = "Elevation unavailable"
                elevation_details = elevation_label
            else:
                elevation_feet = elevation * 3.28084
                elevation_label = (
                    f"Elevation: {elevation:.1f} m ({elevation_feet:.0f} ft)"
                )
                elevation_details = elevation_label
                if resolution is not None:
                    elevation_details += f"<br>Resolution: {resolution:.1f} m"

            tooltip = (
                f"{route['route_id']} to {route['destination_name']} — "
                f"point {location['location_index']}, {reason} — "
                f"{elevation_label}"
            )
            popup_html = (
                f"<b>{route['route_id']} to {route['destination_name']}</b><br>"
                f"Point {location['location_index']} ({reason})<br>"
                f"{elevation_details}"
            )
            folium.CircleMarker(
                point,
                radius=radius,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.8,
                tooltip=tooltip,
                popup=folium.Popup(popup_html, max_width=300),
            ).add_to(layer)

    if mapped_points:
        point_map.fit_bounds(mapped_points)
    folium.LayerControl().add_to(point_map)
    print(
        f"Mapped {point_count} image locations from "
        f"{len(saved_routes)} saved routes. No API calls were made."
    )
    return point_map


## 1. Map saved walking points


In [ ]:
# This cell only reads the saved JSON file; it never calls Google.
walking_points_map = build_point_map(
    saved_data_path=Path("saved_route_points/walking_routes.json"),
    travel_label="Walking",
)
walking_points_map

## 2. Map saved biking points


In [ ]:
# This cell only reads the saved JSON file; it never calls Google.
biking_points_map = build_point_map(
    saved_data_path=Path("saved_route_points/biking_routes.json"),
    travel_label="Biking",
)
biking_points_map